# SILVER PRODUCTS TABLE

#### Cell 1 — read + conform Mockaroo Bronze

In [0]:
from pyspark.sql.functions import to_date, lit, col

mockaroo_silver_df = (
    spark.table("bronze_mockaroo_products_stream")
    .select(
        col("sku").alias("product_key"),
        col("product_name").alias("product_name"),
        col("category").alias("category"),
        col("price").alias("price"),
        col("brand").alias("brand"),
        col("stock_quantity").alias("stock_quantity"),
        to_date(col("created_at"), "M/d/yyyy").alias("created_at"),
        lit(None).cast("double").alias("rating"),
        lit(None).cast("double").alias("discount_percentage"),
        lit(None).cast("string").alias("dimensions_json"),
        lit(None).cast("string").alias("reviews_json"),
        lit("mockaroo").alias("source_system"),
    )
)

#### Cell 2 — read + parse + conform DummyJSON Bronze

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import *

dj_product_payload_schema = StructType([
    StructField("sku", StringType()),
    StructField("title", StringType()),
    StructField("category", StringType()),
    StructField("price", DoubleType()),
    StructField("brand", StringType()),
    StructField("stock", IntegerType()),
    StructField("rating", DoubleType()),
    StructField("discountPercentage", DoubleType()),
    StructField("dimensions", StructType([
        StructField("width", DoubleType()),
        StructField("height", DoubleType()),
        StructField("depth", DoubleType()),
    ])),
    StructField("reviews", ArrayType(StructType([
        StructField("rating", IntegerType()),
        StructField("comment", StringType()),
        StructField("reviewerName", StringType()),
    ]))),
    StructField("meta", StructType([
        StructField("createdAt", StringType()),
    ])),
])

dj_products_parsed = (
    spark.table("bronze_dummyjson_products")
    .withColumn("parsed", from_json(col("raw_payload"), dj_product_payload_schema))
    .select(
        col("parsed.sku").alias("product_key"),
        col("parsed.title").alias("product_name"),
        col("parsed.category").alias("category"),
        col("parsed.price").alias("price"),
        col("parsed.brand").alias("brand"),
        col("parsed.stock").alias("stock_quantity"),
        to_date(col("parsed.meta.createdAt")).alias("created_at"),
        col("parsed.rating").alias("rating"),
        col("parsed.discountPercentage").alias("discount_percentage"),
        col("parsed.dimensions").cast("string").alias("dimensions_json"),
        col("parsed.reviews").cast("string").alias("reviews_json"),
        lit("dummyjson").alias("source_system"),
    )
)

#### Cell 3 — union both into one conformed dataframe

In [0]:
silver_products_source = mockaroo_silver_df.unionByName(dj_products_parsed)
display(silver_products_source.limit(10))

product_key,product_name,category,price,brand,stock_quantity,created_at,rating,discount_percentage,dimensions_json,reviews_json,source_system
1b7f8dbf-1fdf-41ff-96bd-d57bd3033331,Non-Stick Grill Pan,Tools,39.99,Demizz,321,2025-12-24,null,null,null,null,mockaroo
e9c54758-85a8-451c-be15-88fc74fc803c,Outdoor Portable Fire Pit,Toys,149.99,Rhyloo,242,2026-03-02,null,null,null,null,mockaroo
7f84204e-0575-4875-a2df-ceaa25e3242f,Lemon Garlic Shrimp,Beauty,8.99,Dynabox,169,2025-11-01,null,null,null,null,mockaroo
15afe106-efcc-4978-b75b-7917a53503a9,Savory Trail Mix,Music,4.29,Katz,398,2026-03-26,null,null,null,null,mockaroo
cb5e5542-1637-48c1-a0aa-dd62c8646c88,Digital Bullet Journal,Books,24.99,Divanoodle,488,2026-05-07,null,null,null,null,mockaroo
57d04cfe-e0c0-4794-88c9-68fbf0e95f40,Car Seat Organizer,Computers,14.99,Vinder,481,2026-04-12,null,null,null,null,mockaroo
a35410a5-5162-4f69-90a6-5c3c75c445ff,Roasted Red Pepper Dip,Toys,3.29,Topdrive,49,2026-02-12,null,null,null,null,mockaroo
0228e909-9e28-45dc-b8b2-271c3b787ee8,Cooking Utensil Set,Music,24.99,Geba,91,2026-04-16,null,null,null,null,mockaroo
22c60e24-076d-43f9-a178-145e91deb02e,Pumpkin Spice Cookies,Health,3.29,Yakijo,408,2026-02-03,null,null,null,null,mockaroo
91129682-e1ce-4d80-a55b-900e7d3fca85,Stuffed Peppers with Quinoa,Baby,7.99,Rhyzio,436,2025-10-21,null,null,null,null,mockaroo


In [0]:
display(silver_products_source.filter(col("source_system") == "dummyjson").limit(10))

product_key,product_name,category,price,brand,stock_quantity,created_at,rating,discount_percentage,dimensions_json,reviews_json,source_system
BEA-ESS-ESS-001,Essence Mascara Lash Princess,beauty,9.99,Essence,99,2025-04-30,2.56,10.48,"{15.14, 13.08, 22.99}","[{3, Would not recommend!, Eleanor Collins}, {4, Very satisfied!, Lucas Gordon}, {5, Highly impressed!, Eleanor Collins}]",dummyjson
BEA-GLA-EYE-002,Eyeshadow Palette with Mirror,beauty,19.99,Glamour Beauty,34,2025-04-30,2.86,18.19,"{9.26, 22.47, 27.67}","[{5, Great product!, Savannah Gomez}, {4, Awesome product!, Christian Perez}, {1, Poor quality!, Nicholas Bailey}]",dummyjson
BEA-VEL-POW-003,Powder Canister,beauty,14.99,Velvet Touch,89,2025-04-30,4.64,9.84,"{29.27, 27.93, 20.59}","[{4, Would buy again!, Alexander Jones}, {5, Highly impressed!, Elijah Cruz}, {1, Very dissatisfied!, Avery Perez}]",dummyjson
BEA-CHI-LIP-004,Red Lipstick,beauty,12.99,Chic Cosmetics,91,2025-04-30,4.36,12.16,"{18.11, 28.38, 22.17}","[{4, Great product!, Liam Garcia}, {5, Great product!, Ruby Andrews}, {5, Would buy again!, Clara Berry}]",dummyjson
BEA-NAI-NAI-005,Red Nail Polish,beauty,8.99,Nail Couture,79,2025-04-30,4.32,11.44,"{21.63, 16.48, 29.84}","[{2, Poor quality!, Benjamin Wilson}, {5, Great product!, Liam Smith}, {1, Very unhappy with my purchase!, Clara Berry}]",dummyjson
FRA-CAL-CAL-006,Calvin Klein CK One,fragrances,49.99,Calvin Klein,29,2025-04-30,4.37,1.89,"{29.36, 27.76, 20.72}","[{2, Very disappointed!, Layla Young}, {4, Fast shipping!, Daniel Cook}, {3, Not as described!, Jacob Cooper}]",dummyjson
FRA-CHA-CHA-007,Chanel Coco Noir Eau De,fragrances,129.99,Chanel,58,2025-04-30,4.26,16.51,"{24.5, 25.7, 25.98}","[{4, Highly impressed!, Ruby Andrews}, {5, Awesome product!, Leah Henderson}, {5, Very happy with my purchase!, Xavier Wright}]",dummyjson
FRA-DIO-DIO-008,Dior J'adore,fragrances,89.99,Dior,98,2025-04-30,3.8,14.72,"{27.67, 28.28, 11.83}","[{5, Great value for money!, Nicholas Bailey}, {4, Great value for money!, Penelope Harper}, {4, Great product!, Emma Miller}]",dummyjson
FRA-DOL-DOL-009,Dolce Shine Eau de,fragrances,69.99,Dolce & Gabbana,4,2025-04-30,3.96,0.62,"{27.28, 29.88, 18.3}","[{4, Would buy again!, Mateo Bennett}, {4, Highly recommended!, Nolan Gonzalez}, {5, Very happy with my purchase!, Aurora Lawson}]",dummyjson
FRA-GUC-GUC-010,Gucci Bloom Eau de,fragrances,79.99,Gucci,91,2025-04-30,2.74,14.39,"{20.92, 21.68, 11.2}","[{1, Very dissatisfied!, Cameron Perez}, {5, Very happy with my purchase!, Daniel Cook}, {4, Highly impressed!, Addison Wright}]",dummyjson


### Cell 4a — create the silver schema

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS mia_catalog.silver")

DataFrame[]

### Cell 4b — create silver_products inside it, with CDF enabled

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS mia_catalog.silver.silver_products (
        product_key STRING,
        product_name STRING,
        category STRING,
        price DOUBLE,
        brand STRING,
        stock_quantity INT,
        created_at DATE,
        rating DOUBLE,
        discount_percentage DOUBLE,
        dimensions_json STRING,
        reviews_json STRING,
        source_system STRING,
        record_hash STRING,
        last_updated_ts TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("mia_catalog.silver.silver_products created with CDF enabled")

mia_catalog.silver.silver_products created with CDF enabled


In [0]:
spark.sql('select * from mia_catalog.silver.silver_products')

DataFrame[product_key: string, product_name: string, category: string, price: double, brand: string, stock_quantity: int, created_at: date, rating: double, discount_percentage: double, dimensions_json: string, reviews_json: string, source_system: string, record_hash: string, last_updated_ts: timestamp]

### Cell 5 — compute record_hash, then MERGE into the fully-qualified table
#### USING SQL API

In [0]:
from pyspark.sql.functions import concat_ws, md5, current_timestamp

silver_products_with_hash = silver_products_source.withColumn(
    "record_hash",
    md5(concat_ws("|",
        col("product_name"), col("category"), col("price"),
        col("brand"), col("stock_quantity"), col("rating"),
        col("discount_percentage")
    ))
).withColumn("last_updated_ts", current_timestamp())

silver_products_with_hash.createOrReplaceTempView("silver_products_updates")

spark.sql("""
    MERGE INTO mia_catalog.silver.silver_products AS target
    USING silver_products_updates AS source
    ON target.product_key = source.product_key
    WHEN MATCHED AND target.record_hash != source.record_hash THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("MERGE completed")
display(spark.sql("SELECT count(*) FROM mia_catalog.silver.silver_products"))

MERGE completed


count(*)
1195


#### USING PYTHON API

In [0]:
# from delta.tables import DeltaTable

# target_table = DeltaTable.forName(spark, "mia_catalog.silver.silver_products")

# (target_table.alias("target")
#     .merge(silver_products_with_hash.alias("source"), "target.product_key = source.product_key")
#     .whenMatchedUpdateAll(condition="target.record_hash != source.record_hash")
#     .whenNotMatchedInsertAll()
#     .execute())

In [0]:
display(spark.sql("SELECT source_system, count(*) as row_count FROM mia_catalog.silver.silver_products GROUP BY source_system"))

source_system,row_count
mockaroo,1000
dummyjson,194
test,1


#